In [135]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [136]:
df=pd.read_csv('netflix_titles.csv',encoding='latin')
df = df.copy()
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,...,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [137]:
print(df.shape)
print(df.columns)

(8809, 26)
Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15',
       'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19',
       'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23',
       'Unnamed: 24', 'Unnamed: 25'],
      dtype='object')


There are many empty columns


In [138]:
df = df.dropna(axis=1, how='all')

print(df.shape)
print(df.columns)

(8809, 12)
Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')


Removed empty columns

In [139]:
df.isna().sum()

,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3


In [140]:
df['director'] = df['director'].fillna('Unknown')

In [141]:
df['cast'] = df['cast'].fillna(df['cast'].mode()[0])

In [142]:
df['country'] = df['country'].fillna(df['country'].mode()[0])

In [143]:
# Step 1: remove hidden spaces
df['date_added'] = df['date_added'].str.strip()

# Step 2: convert to datetime (no strict format)
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')

# Step 3: check missing values
df['date_added'].isna().sum()

# Step 4 (optional): format as YYYY-MM-DD
df['date_added'] = df['date_added'].dt.strftime('%Y-%m-%d')

df['date_added'] = df['date_added'].fillna(df['date_added'].mode()[0])

In [144]:
df.loc[:,'rating']=df['rating'].mode()[0]

In [145]:
display(df[['type','duration']])

,type,duration
0,Movie,90 min
1,TV Show,2 Seasons
2,TV Show,1 Season
3,TV Show,1 Season
4,TV Show,2 Seasons
...,...,...
8804,Movie,88 min
8805,Movie,88 min
8806,Movie,111 min
8807,TV Show,1 Season


In [146]:
print(df[df['duration'].isna()][['type', 'duration']])

       type duration
5541  Movie      NaN
5794  Movie      NaN
5813  Movie      NaN


In [147]:
df['duration_value'] = df['duration'].str.extract(r'(\d+)')
df['duration_unit'] = df['duration'].str.extract(r'([a-zA-Z]+)')

In [148]:
df['duration_value'] = pd.to_numeric(df['duration_value'], errors='coerce')
movie_median = df[df['type'] == 'Movie']['duration_value'].median()

df.loc[
    (df['type'] == 'Movie') & (df['duration_value'].isna()),
    'duration_value'
] = movie_median

df.loc[
    (df['type'] == 'Movie') & (df['duration_unit'].isna()),
    'duration_unit'
] = 'min'

In [149]:
df['duration_value'].isna().sum()

np.int64(0)

In [150]:
df['duration_unit'] = df['duration_unit'].replace({
    'Season': 'Seasons'
})

In [151]:
if 'duration' in df.columns:
    df.drop(['duration'], axis=1, inplace=True)
if 'duration_value' in df.columns:
    df.drop(['duration_value'], axis=1, inplace=True)

In [152]:
df.isna().sum()

,0
show_id,0
type,0
title,0
director,0
cast,0
country,0
date_added,0
release_year,0
rating,0
listed_in,0


In [153]:
df.head(10)

,show_id,type,title,director,cast,country,date_added,release_year,rating,listed_in,description,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,David Attenborough,United States,2021-09-25,2020,TV-MA,Documentaries,"As her father nears the end of his life, filmm...",min
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",Seasons
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",United States,2021-09-24,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,Seasons
3,s4,TV Show,Jailbirds New Orleans,Unknown,David Attenborough,United States,2021-09-24,2021,TV-MA,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",Seasons
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,Seasons
5,s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, H...",United States,2021-09-24,2021,TV-MA,"TV Dramas, TV Horror, TV Mysteries",The arrival of a charismatic young priest brin...,Seasons
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, JosÃ© Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",United States,2021-09-24,2021,TV-MA,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...,min
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...",2021-09-24,1993,TV-MA,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s...",min
8,s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Ho...",United Kingdom,2021-09-24,2021,TV-MA,"British TV Shows, Reality TV",A talented batch of amateur bakers face off in...,Seasons
9,s10,Movie,The Starling,Theodore Melfi,"Melissa McCarthy, Chris O'Dowd, Kevin Kline, T...",United States,2021-09-24,2021,TV-MA,"Comedies, Dramas",A woman adjusting to life after a loss contend...,min


In [154]:
df['date_added']=pd.to_datetime(df['date_added'])

In [155]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8809 entries, 0 to 8808
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   show_id        8809 non-null   object        
 1   type           8809 non-null   object        
 2   title          8809 non-null   object        
 3   director       8809 non-null   object        
 4   cast           8809 non-null   object        
 5   country        8809 non-null   object        
 6   date_added     8809 non-null   datetime64[ns]
 7   release_year   8809 non-null   int64         
 8   rating         8809 non-null   object        
 9   listed_in      8809 non-null   object        
 10  description    8809 non-null   object        
 11  duration_unit  8809 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(10)
memory usage: 826.0+ KB


In [156]:
df.duplicated().sum()

np.int64(0)

In [157]:
df.describe()

,date_added,release_year
count,8809,8809.000000
mean,2019-05-17 21:56:34.846180096,2014.181292
min,2008-01-01 00:00:00,1925.000000
25%,2018-04-06 00:00:00,2013.000000
50%,2019-07-04 00:00:00,2017.000000
75%,2020-08-19 00:00:00,2019.000000
max,2024-04-05 00:00:00,2024.000000
std,NaN,8.818932


In [158]:
df['date_added'].nunique()

1715